# 04 — Cross-Model Comparison

**Project:** UREP 32-0210-250078 | Crack Classification

Unified comparison of all 5 model architectures on the same held-out test set:

| # | Model | Parameters | Type |
|---|-------|-----------|------|
| 1 | SVM (HOG+LBP+GLCM) | N/A (handcrafted) | Traditional ML |
| 2 | CNN from Scratch | ~500K | Deep Learning |
| 3 | InceptionV3 (Transfer Learning) | ~24M | Transfer Learning |
| 4 | InceptionV3 + CBAM | ~24M+ | Hybrid Attention |
| 5 | YOLOv8s-cls | ~6.4M | Detection-family |

**Metrics:** Accuracy, Precision, Recall, F1-Score (macro/weighted), IoU (per-class + mean)

All models evaluated on identical test set (6,419 images, 6 classes).

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config

OUTPUT_DIR = config.OUTPUT_DIR

# Model names and their metrics file locations
MODELS = {
    "SVM (RBF)": os.path.join(OUTPUT_DIR, "svm", "metrics_svm.json"),
    "Random Forest": os.path.join(OUTPUT_DIR, "svm", "metrics_random_forest.json"),
    "kNN": os.path.join(OUTPUT_DIR, "svm", "metrics_knn.json"),
    "CNN Scratch": os.path.join(OUTPUT_DIR, "cnn", "metrics_cnn.json"),
    "InceptionV3": os.path.join(OUTPUT_DIR, "inceptionv3", "metrics_inceptionv3.json"),
    "InceptionV3+CBAM": os.path.join(OUTPUT_DIR, "cbam", "metrics_cbam.json"),
    "YOLOv8s-cls": os.path.join(OUTPUT_DIR, "yolo", "metrics_yolo.json"),
}

print("Checking for available model results...")
available = {}
for name, path in MODELS.items():
    exists = os.path.exists(path)
    status = "FOUND" if exists else "NOT FOUND"
    print(f"  {name:<20} {status}  ({path})")
    if exists:
        with open(path) as f:
            available[name] = json.load(f)

print(f"\n{len(available)}/{len(MODELS)} models have results.")
if len(available) < 2:
    print("\nWARNING: Need at least 2 models to compare. Run training notebooks first.")

## Master Comparison Table

In [ ]:
if available:
    rows = []
    for name, m in available.items():
        rows.append({
            "Model": name,
            "Accuracy": m["accuracy"],
            "F1 (macro)": m["f1_macro"],
            "F1 (weighted)": m["f1_weighted"],
            "Precision (macro)": m["precision_macro"],
            "Recall (macro)": m["recall_macro"],
            "Mean IoU": m["mean_iou"],
        })

    df = pd.DataFrame(rows).set_index("Model")
    df = df.sort_values("F1 (macro)", ascending=False)

    print("\n" + "="*90)
    print("MASTER COMPARISON TABLE (sorted by F1 macro)")
    print("="*90)
    print(df.to_string(float_format="{:.4f}".format))
    print("="*90)

    # Save to CSV
    csv_path = os.path.join(OUTPUT_DIR, "model_comparison.csv")
    df.to_csv(csv_path)
    print(f"\nComparison table saved to: {csv_path}")

## Per-Class F1 Comparison

In [ ]:
if available:
    # Extract per-class F1 from classification reports
    per_class_rows = []
    for model_name, m in available.items():
        report = m.get("classification_report", {})
        for cls_name in config.CLASS_NAMES:
            if cls_name in report:
                per_class_rows.append({
                    "Model": model_name,
                    "Class": cls_name,
                    "F1": report[cls_name]["f1-score"],
                    "Precision": report[cls_name]["precision"],
                    "Recall": report[cls_name]["recall"],
                })

    if per_class_rows:
        df_pc = pd.DataFrame(per_class_rows)

        # Pivot for grouped bar chart
        df_f1 = df_pc.pivot(index="Class", columns="Model", values="F1")
        df_f1 = df_f1.reindex(config.CLASS_NAMES)  # preserve class order

        fig, ax = plt.subplots(figsize=(14, 6))
        df_f1.plot(kind="bar", ax=ax, width=0.8)
        ax.set_ylabel("F1-Score")
        ax.set_title("Per-Class F1-Score by Model")
        ax.set_xticklabels(config.CLASS_NAMES, rotation=45, ha="right")
        ax.legend(loc="upper right", fontsize=8)
        ax.set_ylim(0, 1.05)
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        save_path = os.path.join(OUTPUT_DIR, "plots", "per_class_f1_comparison.png")
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved to: {save_path}")

## Macro F1 Bar Chart

In [ ]:
if available:
    models = list(available.keys())
    f1_scores = [available[m]["f1_macro"] for m in models]

    # Sort by F1
    sorted_pairs = sorted(zip(models, f1_scores), key=lambda x: x[1], reverse=True)
    models_sorted, f1_sorted = zip(*sorted_pairs)

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(models_sorted)))
    bars = ax.barh(range(len(models_sorted)), f1_sorted, color=colors)
    ax.set_yticks(range(len(models_sorted)))
    ax.set_yticklabels(models_sorted)
    ax.set_xlabel("Macro F1-Score")
    ax.set_title("Model Comparison — Macro F1-Score")
    ax.set_xlim(0, 1.05)
    ax.grid(axis="x", alpha=0.3)

    # Add value labels
    for bar, val in zip(bars, f1_sorted):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f"{val:.4f}", va="center", fontsize=9)

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, "plots", "macro_f1_comparison.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to: {save_path}")

## Per-Class IoU Comparison

In [ ]:
if available:
    iou_rows = []
    for model_name, m in available.items():
        iou_pc = m.get("iou_per_class", {})
        for cls_name in config.CLASS_NAMES:
            if cls_name in iou_pc:
                iou_rows.append({
                    "Model": model_name,
                    "Class": cls_name,
                    "IoU": iou_pc[cls_name],
                })

    if iou_rows:
        df_iou = pd.DataFrame(iou_rows)
        df_iou_pivot = df_iou.pivot(index="Class", columns="Model", values="IoU")
        df_iou_pivot = df_iou_pivot.reindex(config.CLASS_NAMES)

        print("\nPer-Class IoU:")
        print(df_iou_pivot.to_string(float_format="{:.4f}".format))

        # Mean IoU bar chart
        mean_ious = {name: m["mean_iou"] for name, m in available.items()}
        sorted_iou = sorted(mean_ious.items(), key=lambda x: x[1], reverse=True)

        fig, ax = plt.subplots(figsize=(10, 5))
        names, values = zip(*sorted_iou)
        colors = plt.cm.plasma(np.linspace(0.3, 0.9, len(names)))
        bars = ax.barh(range(len(names)), values, color=colors)
        ax.set_yticks(range(len(names)))
        ax.set_yticklabels(names)
        ax.set_xlabel("Mean IoU")
        ax.set_title("Model Comparison — Mean IoU")
        ax.set_xlim(0, 1.05)
        ax.grid(axis="x", alpha=0.3)

        for bar, val in zip(bars, values):
            ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                    f"{val:.4f}", va="center", fontsize=9)

        plt.tight_layout()
        save_path = os.path.join(OUTPUT_DIR, "plots", "mean_iou_comparison.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved to: {save_path}")

## Summary Table for Paper

Ready-to-use table for the research paper.

In [ ]:
if available:
    print("\n" + "="*100)
    print("TABLE FOR PAPER — Model Performance Comparison on Held-Out Test Set (6,419 images)")
    print("="*100)
    header = f"{'Model':<22} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1 (macro)':>11} {'F1 (wtd)':>10} {'Mean IoU':>10}"
    print(header)
    print("-" * len(header))

    # Sort by F1 macro descending
    for name in sorted(available.keys(), key=lambda n: available[n]["f1_macro"], reverse=True):
        m = available[name]
        print(f"{name:<22} {m['accuracy']:>10.4f} {m['precision_macro']:>10.4f} "
              f"{m['recall_macro']:>10.4f} {m['f1_macro']:>11.4f} {m['f1_weighted']:>10.4f} {m['mean_iou']:>10.4f}")

    print("="*100)
    print("\nAll models evaluated on identical test set with identical metrics.")
    print("Primary metric: Macro F1-Score (equally weights all 6 classes).")